# Стадия 2. Применим ли самый простой линейный метод (оценка по числу)

**Источник данных:** `train_model.parquet` (стадия 1); первичный датасет — Kaggle S3E26, первоисточник — UCI Cirrhosis.
**Предыдущий этап:** `Stage_1_analysis.ipynb` — анализ данных и отбор 15 признаков.

**О чём эта стадия.** Самый простой способ предсказать исход — посчитать его как взвешенную сумму анализов пациента (линейная регрессия, «оценка по числу»). Метод быстр и прозрачен, поэтому его пробуют первым. Здесь мы проверяем не качество, а саму возможность использовать линейную регрессию для задачи классификации.

**Почему проверка делается до обучения.** У линейного метода есть набор условий («предпосылок»), при которых он корректен. Если условие грубо нарушено, результат моделирования известен заранее, и тратить цикл обучения бессмысленно. Проверка занимает секунды и даёт тот же управленческий вывод, что полный цикл с метриками.

**Что берём в работу.** Матрицу из **15 отобранных признаков** и бинарную цель `Event` — файл `train_model.parquet`, подготовленный на стадии 1. Бинарная цель: 0 — пациент жив со своей печенью, 1 — трансплантация или смерть. Период наблюдения (`N_Days`) в матрицу не входит: как показано в блоке 3.8 стадии 1, период становится известен только после наступления исхода. Дубликаты показателей тоже исключены — действует правило «одна информация — один столбец».

**Постановка проверки.** Бинарная цель уже закодирована как 0 и 1. Кодировка ближе к реальности, чем три класса, но остаётся вопрос: может ли линейная регрессия выдавать осмысленные значения между 0 и 1, или она будет уходить за границы диапазона? Первый пункт проверки отвечает на этот вопрос.


## **1. Импорты и настройки**

In [1]:
# Сторонние библиотеки
import os

import numpy as np
import pandas as pd

pd.set_option('display.width', 200)

## **2. Загрузка данных**

Данные берутся из `data/processed/train_model.parquet` — это итоговый набор стадии 1 (блок 3.14): 15 признаков модели плюс бинарная цель `Event`, 7 905 строк после очистки. Повторная очистка и отбор не требуются — решения приняты и обоснованы на стадии 1.


In [2]:
# Корень проекта: папка с data/, найденная вверх по дереву от текущей директории
def find_project_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(d, 'data')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            raise FileNotFoundError('Папка data/ не найдена вверх по дереву от ' + d)
        d = parent


PROJECT_ROOT = find_project_root()
df = pd.read_parquet(os.path.join(PROJECT_ROOT, 'data', 'processed', 'train_model.parquet'))
print(f'Строк: {len(df)}, столбцов: {df.shape[1]} (15 признаков + цель)')
df.head(3)

Строк: 7905, столбцов: 16 (15 признаков + цель)


,log_Bilirubin,log_Copper,log_Alk_Phos,log_SGOT,log_Cholesterol,log_Tryglicerides,Albumin,Platelets,Prothrombin,Age_years,Stage,Edema_num,Sex_M,Hepatomegaly_Y,Spiders_Y,Event
0,1.193922,5.153292,7.379008,5.197391,5.758902,4.158883,3.35,394.0,9.7,59.0,3.0,0.0,1,0,0,1
1,0.641854,4.158883,7.273093,4.911551,5.899897,4.488636,3.54,361.0,11.0,52.7,3.0,0.0,0,0,0,0
2,1.458615,4.882802,6.937314,4.790404,5.703782,3.931826,3.55,199.0,11.7,37.6,4.0,1.0,0,1,1,1


### Целевая переменная

In [3]:
# Бинарная цель: 0 — жив со своей печенью, 1 — трансплантация или смерть
y = df['Event'].astype(int)

print('Распределение классов бинарной цели:')
display(y.value_counts(normalize=True).sort_index().rename('доля').round(4).to_frame())


Распределение классов бинарной цели:


,доля
Event,
0,0.6281
1,0.3719


Матрица признаков — это все столбцы файла, кроме цели `Event`. Никакого дополнительного кодирования не нужно: бинарные признаки (мужской пол, увеличенная печень, сосудистые «паучки») уже приведены к 0/1 на стадии 1, порядковая шкала отёков — к числам 0 / 0.5 / 1, скошенные лабораторные показатели — к логарифмам.

Напоминание, почему в наборе нет остальных столбцов (обоснование — блоки 2.5, 3.8, 3.10, 3.11 стадии 1):

| Причина исключения | Столбцы |
|---|---|
| Дублируют выбранный вариант одного и того же факта | `Bilirubin`, `Copper`, `Alk_Phos`, `SGOT`, `Cholesterol`, `Tryglicerides`, `Age`, `Edema`, `is_severe`, `Ascites` |
| Известен только после наступления исхода | `N_Days` (период наблюдения) |
| Не связан с исходом / технический | `Drug`, `id` |
| Прямая утечка цели | `is_dead` |


In [4]:
X = df.drop(columns=['Event']).astype(float)

print(f'Матрица признаков: {X.shape[0]} строк x {X.shape[1]} столбцов')
print('Столбцы:', list(X.columns))


Матрица признаков: 7905 строк x 15 столбцов
Столбцы: ['log_Bilirubin', 'log_Copper', 'log_Alk_Phos', 'log_SGOT', 'log_Cholesterol', 'log_Tryglicerides', 'Albumin', 'Platelets', 'Prothrombin', 'Age_years', 'Stage', 'Edema_num', 'Sex_M', 'Hepatomegaly_Y', 'Spiders_Y']


## **4. Быстрая оценка применимости линейной регрессии**

У линейной регрессии есть набор условий, которые можно проверить за секунды — **без** обучения модели, разбиения на части и анализа остатков. Если условие грубо нарушено, результат моделирования предсказуем заранее. Проверяем четыре вещи на всей выборке (модели нет, поэтому подглядывания в ответ не грозит):

1. **Тип и распределение цели** — линейная регрессия рассчитана на непрерывную величину на всей числовой прямой. Для бинарной цели это значит, что предсказания могут выйти за диапазон [0, 1], где лежат истинные значения.
2. **Наличие линейной связи** «признак ↔ цель» (корреляция Пирсона, которая для бинарной цели совпадает с точечно-биссериальной корреляцией).
3. **Линейность этой связи** — сравнение Пирсона и Спирмена.
4. **Мультиколлинеарность** между признаками — близкие пары делают веса нестабильными.

Дополнительно подгоняем линейную регрессию на всей выборке — не ради качества, а чтобы посчитать, какая доля её предсказаний выходит за диапазон [0, 1].

> **Пирсон против Спирмена.** Пирсон меряет силу *линейной* связи, Спирмен — силу *монотонной* (любой возрастающей/убывающей). Если Спирмен заметно больше Пирсона, связь есть, но она нелинейна — линейная регрессия её недоиспользует.


In [5]:
# Проверка 1. Тип и распределение цели
n_classes = y.nunique()
print(f'Целевая переменная принимает {n_classes} значения: {dict(y.value_counts().sort_index())}')

if n_classes == 2:
    print('=> Цель ДИСКРЕТНАЯ (два значения 0/1). Линейная регрессия формально не предназначена для такой цели:')
    print('   она не ограничивает предсказания диапазоном [0, 1], поэтому часть предсказаний')
    print('   выйдет за границы — значения ниже 0 или выше 1 не имеют смысла вероятностей.')
    print('   Это ограничение метода, а не данных.')
else:
    print('=> Цель почти непрерывная — предпосылка линейной регрессии выполнена.')

# Подгонка на всей выборке — только для подсчёта выхода за диапазон [0, 1]
from sklearn.linear_model import LinearRegression
lr = LinearRegression().fit(X, y)
preds = lr.predict(X)
outside = (preds < 0) | (preds > 1)
print(f'\nДиапазон предсказаний линейной регрессии: [{preds.min():.3f}, {preds.max():.3f}]')
print(f'Доля предсказаний вне [0, 1]: {outside.mean():.1%} ({outside.sum()} из {len(y)})')
print('=> Предсказания за пределами диапазона не имеют смысла вероятностей.')


Целевая переменная принимает 2 значения: {0: np.int64(4965), 1: np.int64(2940)}
=> Цель ДИСКРЕТНАЯ (два значения 0/1). Линейная регрессия формально не предназначена для такой цели:
   она не ограничивает предсказания диапазоном [0, 1], поэтому часть предсказаний
   выйдет за границы — значения ниже 0 или выше 1 не имеют смысла вероятностей.
   Это ограничение метода, а не данных.



Диапазон предсказаний линейной регрессии: [-0.223, 1.628]
Доля предсказаний вне [0, 1]: 10.8% (850 из 7905)
=> Предсказания за пределами диапазона не имеют смысла вероятностей.


In [6]:
# Проверка 2. Линейная связь каждого признака с целью (Пирсон / точечно-биссериальная корреляция)
corr_with_target = X.corrwith(y).sort_values(key=abs, ascending=False)
print('Топ-8 линейных связей с целью:')
display(corr_with_target.head(8).round(3).to_frame('r (Пирсон)'))

max_abs = corr_with_target.abs().max()
if max_abs < 0.2:
    verdict2 = 'слабые: почти все |r| < 0.2 — линейной модели брать практически нечего'
elif max_abs < 0.4:
    verdict2 = 'умеренные: |r| ~ 0.2-0.4 — линейная связь есть, но сигнал не сильный'
else:
    verdict2 = 'умеренные: |r| ~ 0.3-0.5 — линейная связь есть, но сигнал не сильный'
print(f'\n=> Связи с целью: {verdict2}. Максимум |r| = {max_abs:.2f} ({corr_with_target.abs().idxmax()}).')


Топ-8 линейных связей с целью:


,r (Пирсон)
log_Bilirubin,0.531
log_Copper,0.415
Hepatomegaly_Y,0.392
Prothrombin,0.373
Stage,0.342
log_SGOT,0.331
Edema_num,0.322
Spiders_Y,0.319



=> Связи с целью: умеренные: |r| ~ 0.3-0.5 — линейная связь есть, но сигнал не сильный. Максимум |r| = 0.53 (log_Bilirubin).


In [7]:
# Проверка 3. Линейность связи: Пирсон против Спирмена для сильнейших признаков
top_feats = corr_with_target.head(6).index
lin_check = pd.DataFrame({
    'Пирсон': [X[f].corr(y) for f in top_feats],
    'Спирмен': [X[f].corr(y, method='spearman') for f in top_feats],
}, index=top_feats)
lin_check['разрыв'] = (lin_check['Спирмен'] - lin_check['Пирсон']).abs()
display(lin_check.round(3))

print('=> Разрыв у всех сильнейших признаков не больше 0.023: связи почти линейны.')
print('   Это ожидаемо: скошенные показатели уже приведены к логарифмической шкале')
print('   на стадии 1 (блок 2.5), и именно log-варианты лежат в матрице признаков.')
print('   Для сравнения: неубранный исходный билирубин давал разрыв 0.122 —')
print('   проверка 3 на исходных данных выявила бы нелинейность.')


,Пирсон,Спирмен,разрыв
log_Bilirubin,0.531,0.542,0.010
log_Copper,0.415,0.439,0.023
Hepatomegaly_Y,0.392,0.392,0.000
Prothrombin,0.373,0.371,0.002
Stage,0.342,0.358,0.015
log_SGOT,0.331,0.343,0.011


=> Разрыв у всех сильнейших признаков не больше 0.023: связи почти линейны.
   Это ожидаемо: скошенные показатели уже приведены к логарифмической шкале
   на стадии 1 (блок 2.5), и именно log-варианты лежат в матрице признаков.
   Для сравнения: неубранный исходный билирубин давал разрыв 0.122 —
   проверка 3 на исходных данных выявила бы нелинейность.


In [8]:
# Проверка 4. Мультиколлинеарность между признаками
feat_corr = X.corr()
mask = np.triu(np.ones_like(feat_corr, dtype=bool), k=1)
pairs = feat_corr.where(mask).stack().sort_values(key=abs, ascending=False)

print('Топ-6 пар признаков по |r|:')
display(pairs.head(6).round(3).to_frame('|r|'))

n_08 = int((pairs.abs() > 0.8).sum())
n_06 = int((pairs.abs() > 0.6).sum())
print(f'\n=> Пар с |r| > 0.8: {n_08}. Пар с |r| > 0.6: {n_06}.')
print(f'Сильнейшая пара: «{pairs.index[0][0]} ↔ {pairs.index[0][1]}» |r| = {pairs.iloc[0]:.3f}')
print('Это клинически осмысленное сочетание (билирубин и медь растут при поражении')
print('печени), а не дублирование. Предпосылка выполнена: проблема начинается от ~0.8.')
print('\nНапоминание: дубликаты (показатель и его логарифм, асцит и отёки) удалены')
print('на стадии 1 — поэтому здесь видны только настоящие связи данных.')


Топ-6 пар признаков по |r|:


,,|r|
log_Bilirubin,log_Copper,0.538
Stage,Hepatomegaly_Y,0.498
log_Bilirubin,log_SGOT,0.464
log_Copper,log_SGOT,0.403
log_Bilirubin,Hepatomegaly_Y,0.387
log_SGOT,log_Cholesterol,0.352



=> Пар с |r| > 0.8: 0. Пар с |r| > 0.6: 0.
Сильнейшая пара: «log_Bilirubin ↔ log_Copper» |r| = 0.538
Это клинически осмысленное сочетание (билирубин и медь растут при поражении
печени), а не дублирование. Предпосылка выполнена: проблема начинается от ~0.8.

Напоминание: дубликаты (показатель и его логарифм, асцит и отёки) удалены
на стадии 1 — поэтому здесь видны только настоящие связи данных.


### Вердикт по предпосылкам (до всякого обучения модели)

По четырём быстрым проверкам заранее видно:

| Предпосылка | Что показали метрики | Вывод для линейной регрессии |
|---|---|---|
| Предсказания в диапазоне [0, 1] | 10,8 % предсказаний вне [0, 1], диапазон [-0.223, 1.628] | **Грубо нарушена**: линейная регрессия не ограничивает выход, часть предсказаний бессмысленна |
| Линейная связь с целью | максимум \|r\| ≈ 0.53 (`log_Bilirubin`) | Сигнал есть — признаки информатны, но связность умеренная |
| Линейность связи | разрыв Пирсона и Спирмена ≤ 0.023 у сильнейших признаков | Предпосылка выполнена — спасибо лог-преобразованиям и отбору стадии 1 |
| Мультиколлинеарность | ни одной пары \|r\| > 0.6, сильнейшая `log_Bilirubin ↔ log_Copper` = 0.538 | Предпосылка выполнена |

С признаками всё в порядке: связь почти линейна, близких пар нет — это хороший результат для самих данных. Нарушено одно, но неустранимое: цель принимает два значения, а линейная регрессия об этом не знает и не может знать — ограничение на диапазон [0, 1] противоречит самой идее взвешенной суммы. К нему добавляется второе следствие бинарной цели: остатки такой цели гетероскедастичны, их разброс зависит от предсказанного значения (при вероятности 0.5 дисперсия максимальна, у краёв стремится к нулю), тогда как линейная регрессия предполагает постоянный разброс ошибки. Обе проблемы лечатся сменой модели, а не обработкой данных.

> **Вывод по методике:** линейную регрессию для задач классификации не используют. Корректный инструмент для предсказания `Event` — логистическая регрессия (и далее ансамбли), с метриками классификации (ROC-AUC, лог-потеря, F-мера). Быстрая проверка заняла секунды и дала тот же управленческий вывод, что занял бы полный цикл моделирования.


## **5. Выводы**

### Ответ на вопрос задачи

**Можно ли применить линейную регрессию (оценку по числу) для предсказания бинарного события (выживание со своей печенью)?** Проверка предпосылок даёт отрицательный ответ, и он следует из структуры задачи, а не из качества данных:

1. **Цель не является непрерывной.** `Event` принимает два значения: 0 (жив) и 1 (трансплантация или смерть). Линейная регрессия по природе выдаёт любое вещественное число: предсказания вроде −0.2 или 1.6 не соответствуют ни одному исходу. В задаче классификации такие числа не имеют смысла — пациент либо жив, либо нет.

2. **Метрики качества регрессии были бы неинтерпретируемы.** Коэффициент детерминации (R²), средняя абсолютная ошибка (MAE) и среднеквадратичная ошибка (RMSE) измеряют ошибку в произвольных единицах; «ошибка 0.4» не отвечает на практический вопрос «выживёт ли пациент». Для классификации нужны доля верных ответов, ROC-AUC, логарифмическая потеря и F-мера.

3. **Сами данные для моделирования пригодны.** Признаки информативны (максимум связи с целью |r| = 0.53 у `log_Bilirubin`), связи почти линейны (разрыв Пирсона и Спирмена ≤ 0.023), близких пар между признаками нет (сильнейшая — `log_Bilirubin ↔ log_Copper` = 0.538, клинически осмысленное сочетание маркёров поражения печени). Вывод «нет» относится к выбору метода, а не к датасету.

4. **Практический итог метода:** отказ от линейной регрессии здесь принят за секунды корреляционных расчётов, без единого обучения модели. Проверка предпосылок сэкономила цикл моделирования и защитила от некорректной интерпретации его результатов.

### Глоссарий терминов

| Термин | Что означает | Зачем нужен |
|--------|-------------|-------------|
| **Целевая переменная** | То, что модель пытается предсказать (здесь — `Event`: 0 или 1) | Без неё модель не знает, чему учиться |
| **Признаки** | Входы модели: лабораторные показатели, стадия, демография (15 столбцов) | Информация, на основе которой строится предсказание |
| **Бинарная цель** | Цель задаётся двумя классами: событие и отсутствие события | Определяет выбор класса моделей: нужны классификаторы, а не регрессии |
| **Предпосылка модели** | Свойство данных, при котором метод корректен (непрерывность цели, линейность связи, отсутствие близких признаков) | Проверка предпосылок отвечает на вопрос «можно ли применять метод» до вопроса «насколько качественно» |
| **Корреляция Пирсона** | Сила *линейной* связи, от −1 до 1 | Показывает, есть ли что «вытягивать» линейной модели |
| **Точечно-биссериальная корреляция** | Корреляция Пирсона, когда одна из переменных бинарна | Позволяет измерить линейную связь признака с `Event` |
| **Корреляция Спирмена** | Сила *монотонной* связи (по порядку значений) | Сравнение с Пирсоном выявляет нелинейность: большой разрыв = связь есть, но не линейная |
| **Мультиколлинеарность** | Высокая корреляция между признаками | Делает веса модели нестабильными: знак может «переворачиваться» |
| **Гетероскедастичность остатков** | Разброс ошибки зависит от предсказанного значения | У бинарной цели дисперсия максимальна при вероятности 0.5 и падает к краям — линейная модель этого не учитывает |
| **Доля объяснённого разброса (R²)** | Какая часть колебаний цели описывается моделью, от 0 до 1 | Мера качества регрессии; для категорий не применима |
| **Логарифмическая потеря (log-loss)** | Штраф за уверенность в неверном ответе классификатора | Главная метрика качества вероятностного прогноза |
| **Утечка цели** | В признаки попадает информация, прямо вычисляемая из цели (`is_dead` — это `Status == 'D'`) | Модель «предсказывает» известное, метрики становятся фиктивными |
| **Период наблюдения (`N_Days`)** | Сколько дней пациент состоял под наблюдением до развязки | Известен только после исхода — на стадии 1 исключён из признаков как следствие, а не причина |
